# 03 — Question 2 : l'exposition à l'IA améliore-t-elle la prédiction des salaires ?

**Question :** Does adding the AI exposure score to a baseline wage model reduce prediction error on a held-out test set, split by individual? If so, is the gain specific to AI exposure, or does it come from finer occupational detail?

**Règle :** aucune ligne individuelle affichée, seulement des RMSE, des moyennes et des comptages.

**Méthodes du cours :** `train_test_split` et `LinearRegression` (Week 6), `RandomForestRegressor` et `GridSearchCV` (Week 7), `get_dummies` (Week 2), RMSE (Week 6).
**Au-delà du cours (signalé) :** `GroupKFold` pour la validation croisée par personne (étape 6).

| Modèle | Variables | Ce qu'il teste |
|---|---|---|
| A | Baseline : âge, sexe, formation, région, année, secteur | Référence |
| B | A + exposition | L'exposition réduit-elle l'erreur ? |
| C | A + placebo (scores mélangés entre métiers) | Même finesse, sans le sens « IA » |
| D | A + grand groupe ISCO (1 chiffre) | Information grossière sur le métier |
| E | D + exposition | L'exposition apporte-t-elle quelque chose à grand groupe égal ? |
| F | A + ISCO 2 chiffres | Métier plus fin, sans IA |
| G | F + exposition | L'exposition apporte-t-elle quelque chose au-delà de catégories de métier **fines** ? |

**Pourquoi le modèle G ?** La question demande si le gain est spécifique à l'IA ou s'il vient d'un détail plus fin sur le métier.
Le test E contre D y répond seulement pour les **grands groupes** (9 catégories). Or, un score qui n'ajoute rien au-delà de 9 catégories pourrait encore ajouter quelque chose au-delà de catégories plus fines, et inversement.
G contre F répète donc le même test avec les **44 groupes à 2 chiffres** : si l'exposition n'apporte rien non plus à ce niveau, on peut conclure qu'elle n'ajoute pas d'information au-delà des catégories de métier standard, et pas seulement au-delà des grands groupes.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

DATA_PATH = "../Data/private/shp_isco4_ilo.parquet"

## 1. Charger les données
On retire les 80 lignes sans score d'exposition et la catégorie `sex = 3` (22 lignes, apparue en 2024), comme prévu dans le cleaning.

In [ ]:
df = pd.read_parquet(DATA_PATH)
df = df[df["exposure_source"] != "aucun"]
df = df[df["sex"].isin([1, 2])]

print("Lignes :", df.shape[0], "| Personnes :", df["idpers"].nunique())
print(df["exposure_source"].value_counts())

## 2. Préparer les variables
- `age2` = âge au carré : le salaire augmente avec l'âge puis plafonne (profil concave). Une régression linéaire a besoin de ce terme pour capter la courbure.
- Variables catégorielles → dummies avec `get_dummies(drop_first=True)` pour éviter le piège des dummies (Week 6).
- `isco1` et `isco2` : premiers chiffres du code ISCO 4 (texte).

In [ ]:
df["age2"] = df["age"] ** 2
df["isco1"] = df["isco4"].str[:1]
df["isco2"] = df["isco4"].str[:2]

# Catégories codées en chiffres -> texte, pour que get_dummies les traite comme des catégories
for col in ["sex", "isced", "region", "year", "noga2m"]:
    df[col] = df[col].astype("Int64").astype(str)

dummies = pd.get_dummies(df[["sex", "isced", "region", "year", "noga2m"]], drop_first=True, dtype=int)
d_isco1 = pd.get_dummies(df["isco1"], prefix="isco1", drop_first=True, dtype=int)
d_isco2 = pd.get_dummies(df["isco2"], prefix="isco2", drop_first=True, dtype=int)

BASE = ["age", "age2"] + list(dummies.columns)
df = pd.concat([df, dummies, d_isco1, d_isco2], axis=1)

print("Variables baseline :", len(BASE))
print("Dummies ISCO 1 chiffre :", d_isco1.shape[1], "| ISCO 2 chiffres :", d_isco2.shape[1])

## 3. Score placebo
On garde **un chiffre par métier**, mais on **mélange les scores entre métiers** : un développeur reçoit par exemple le score d'un boulanger.
Le placebo a donc exactement la même finesse que l'exposition, mais plus aucun sens « IA ».

In [ ]:
scores_metier = df.groupby("isco4")["exposure_main"].first()

rng = np.random.default_rng(42)
scores_melanges = pd.Series(rng.permutation(scores_metier.values), index=scores_metier.index)
df["placebo"] = df["isco4"].map(scores_melanges)

print("Corrélation exposition / placebo :", round(df["exposure_main"].corr(df["placebo"]), 3))   # proche de 0

## 4. Les six jeux de variables

In [ ]:
MODELES = {
    "A_baseline":          BASE,
    "B_exposition":        BASE + ["exposure_main"],
    "C_placebo":           BASE + ["placebo"],
    "D_isco1":             BASE + list(d_isco1.columns),
    "E_isco1_exposition":  BASE + list(d_isco1.columns) + ["exposure_main"],
    "F_isco2":             BASE + list(d_isco2.columns),
    "G_isco2_exposition":  BASE + list(d_isco2.columns) + ["exposure_main"],
}
y = df["log_income"]

## 5. Split par personne
Une personne apparaît jusqu'à 9 fois. Si ses années étaient réparties entre train et test, le modèle la « reconnaîtrait » et l'erreur de test serait trop optimiste.

On applique donc `train_test_split` (Week 6) **à la liste des personnes**, puis on garde toutes les lignes de chaque personne du même côté.

In [ ]:
def split_par_personne(df, seed):
    personnes = df["idpers"].unique()
    pers_train, pers_test = train_test_split(personnes, test_size=0.2, random_state=seed)
    return df["idpers"].isin(pers_train), df["idpers"].isin(pers_test)

train, test = split_par_personne(df, seed=42)
print("Lignes train :", train.sum(), "| test :", test.sum())
print("Personnes en commun :", len(set(df.loc[train, "idpers"]) & set(df.loc[test, "idpers"])))   # doit être 0

## 6. Choisir les hyperparamètres de la random forest
`GridSearchCV` comme en Week 7, **une seule fois**, sur le modèle B et sur le train uniquement.
On garde ensuite les mêmes hyperparamètres pour tous les modèles : la comparaison porte alors seulement sur les variables.

**Au-delà du cours :** `cv=GroupKFold(5)` avec `groups=idpers` fait aussi la validation croisée **par personne**. Avec le `cv=5` par défaut, une même personne pourrait se retrouver dans deux plis.

In [ ]:
param_grid = {"max_depth": [10, 20, None], "min_samples_leaf": [5, 20, 50]}

grid = GridSearchCV(
    RandomForestRegressor(n_estimators=100, max_features=0.5, random_state=42, n_jobs=-1),
    param_grid,
    cv=GroupKFold(n_splits=5),
    scoring="neg_root_mean_squared_error",
)
grid.fit(df.loc[train, MODELES["B_exposition"]], y[train], groups=df.loc[train, "idpers"])

print("Meilleurs hyperparamètres :", grid.best_params_)
print("RMSE validation croisée :", round(-grid.best_score_, 4))

## 7. Entraîner et évaluer les 14 modèles (7 jeux × 2 algorithmes)
Métrique : **RMSE sur le test**, en log du salaire. Une RMSE de 0,30 correspond à une erreur typique d'environ 30 %.

In [ ]:
def evaluer(train, test):
    resultats = []
    for nom, variables in MODELES.items():
        X_train, X_test = df.loc[train, variables], df.loc[test, variables]

        lr = LinearRegression().fit(X_train, y[train])
        rf = RandomForestRegressor(n_estimators=200, max_features=0.5, random_state=42,
                                   n_jobs=-1, **grid.best_params_).fit(X_train, y[train])

        resultats.append({"modele": nom,
                          "RMSE_lineaire": root_mean_squared_error(y[test], lr.predict(X_test)),
                          "RMSE_foret": root_mean_squared_error(y[test], rf.predict(X_test))})
    return pd.DataFrame(resultats).set_index("modele")

res = evaluer(train, test)

# Gain par rapport à la baseline, en % de la RMSE
for algo in ["RMSE_lineaire", "RMSE_foret"]:
    res["gain_" + algo[5:] + "_%"] = (1 - res[algo] / res.loc["A_baseline", algo]) * 100

res.round(4)

## 8. Robustesse : 10 splits différents
Un seul split peut donner un résultat dû au hasard (quelles personnes tombent dans le test).
On répète avec 10 tirages et on regarde la **moyenne et l'écart-type du gain**. Un gain plus petit que son écart-type n'est pas fiable.

⏱ Cette cellule prend environ 10 à 15 minutes (140 modèles).

In [ ]:
tous = []
for seed in range(10):
    tr, te = split_par_personne(df, seed)
    r = evaluer(tr, te)
    for algo in ["RMSE_lineaire", "RMSE_foret"]:
        r["gain_" + algo[5:] + "_%"] = (1 - r[algo] / r.loc["A_baseline", algo]) * 100
    r["seed"] = seed
    tous.append(r.reset_index())

tous = pd.concat(tous)
resume = tous.groupby("modele")[["gain_lineaire_%", "gain_foret_%"]].agg(["mean", "std"]).round(2)
resume

## 9. Lecture des résultats

| Comparaison | Si… | Alors… |
|---|---|---|
| **B vs A** | gain de B > 0, nettement au-dessus de son écart-type | l'exposition réduit l'erreur de prédiction |
| **B vs C** | B ≫ C | le gain vient du **contenu** du score (le sens « IA »), pas seulement de sa finesse |
| | B ≈ C | n'importe quel chiffre par métier ferait aussi bien → c'est de l'**information sur le métier** |
| **E vs D** | E > D | à grand groupe égal, l'exposition apporte encore quelque chose (**test clé**) |
| | E ≈ D | l'exposition ne fait que résumer le grand groupe de métier |
| **G vs F** | G > F | même avec 44 groupes de métiers, l'exposition apporte encore quelque chose |
| | G ≈ F | l'exposition n'ajoute rien au-delà des catégories de métier fines |
| **B vs F** | F ≫ B | un métier plus fin, sans IA, prédit mieux → le gain de B vient surtout de la finesse du métier |

**Pour E vs D et G vs F, on lit d'abord le modèle linéaire** : en forêt, les hyperparamètres ont été choisis sur le modèle B, ce qui peut avantager les modèles avec exposition.

**Attention, la random forest et le placebo :** une forêt peut se servir d'un chiffre par métier comme d'un **identifiant** du métier, quel que soit son sens. Le placebo C peut donc aider la forêt sans aider la régression linéaire. Le test B vs C est plus parlant avec le modèle **linéaire**, où un score sans lien avec le salaire ne peut pas aider.

**Ce que ça ne dit pas :** un gain de prédiction n'est pas un effet causal. Les métiers exposés sont aussi ceux qui exigent plus de formation, et la question 2 ne dit pas que l'IA *fait* monter ou baisser les salaires.

## 10. Conclusion

**Answer to Q2.** Adding the AI exposure score to the baseline wage model reduces test RMSE on held-out individuals by 0.56% (SD 0.22) with linear regression and 2.51% (SD 0.42) with a random forest (mean over 10 person-level splits). The gain is not an artefact of granularity: a placebo score, which keeps one value per occupation but shuffles values across occupations, yields almost no improvement (0.02% and 0.29%).

However, the gain is occupational information rather than AI-specific information. Occupational categories alone reduce RMSE far more (4.70% with 1-digit ISCO, 5.05% with 2-digit ISCO, linear model), and once they are included, the exposure score adds nothing (+0.03 percentage points at both levels). In the random forest, exposure still adds about 1.4 to 2.0 points beyond occupational categories, but this reflects the forest's difficulty with many occupation dummies: the 2-digit model performs worse than the 1-digit model, and the best forest model (4.44%) remains below the best linear model (5.08%). Exposure therefore acts as a compact numerical summary of occupation, ordered in a way that correlates with wages.

**Interpretation.** AI exposure is a useful summary of occupation for wage prediction, but it carries no predictive information beyond standard occupational categories. This is consistent with exposure reflecting the skill content of occupations. These are predictive associations, not causal effects.

**Limitations.** (1) Overall gains are small: the best model reduces RMSE by about 5% (baseline RMSE ≈ 0.365 in log wage). (2) Random forest hyperparameters were tuned once on model B and applied to all models, which may favour models including exposure; the linear model is therefore the reference for comparisons D–E and F–G. (3) The placebo was drawn once (correlation with true exposure: −0.08).